# Random Forest Experiments (Parts B and C)

This notebook runs the requested experiments on Heart Disease UCI and Intel Image Classification. Part A is not repeated. Run the cells from top to bottom.

**Before starting:** The Heart Disease data is downloaded by the notebook. Download the [Intel Image Classification dataset on Kaggle](https://www.kaggle.com/datasets/puneet6060/intel-image-classification), extract it, and put its `seg_train/seg_train` folder at `lab-assignment/intel-image-classification/seg_train/seg_train`. This notebook samples up to 150 images per class to keep beginner computers from taking too long; set `MAX_IMAGES_PER_CLASS = None` below to use all training images.


## Short paper summary

Breiman's 2001 paper presents Random Forests as collections of randomized decision trees. Bootstrap samples provide different training data for trees, and randomly selected features at each split make trees less alike. Voting across trees reduces the effect of an individual tree's mistakes. Breiman also discusses the strength of trees and their correlation as important factors in generalization, and describes out-of-bag samples as a way to estimate error without a separate validation set.


In [3]:
from pathlib import Path
import io
import time
import zipfile
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TREE_COUNTS = [1, 10, 50, 100, 300]

# Find the assignment folder whether the notebook is opened from the workspace or its subfolder.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'data.csv').exists() and (NOTEBOOK_DIR / 'lab-assignment').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'lab-assignment'
RESULTS_DIR = NOTEBOOK_DIR / 'random_forest_results'
RESULTS_DIR.mkdir(exist_ok=True)
print('Saving plots and result tables to:', RESULTS_DIR.resolve())

Saving plots and result tables to: E:\python-learning\lab-assignment\random_forest_results


## 1. Heart Disease UCI dataset

The Cleveland subset contains patient measurements and a diagnosis field. We convert the diagnosis to two classes (0 = no disease, 1 = disease present), convert missing-value markers to missing data, remove incomplete rows, and use a stratified 80/20 train/test split. Features are standardized for the SVM; tree models use the original numeric features.


In [4]:
heart_url = 'https://archive.ics.uci.edu/static/public/45/heart+disease.zip'
heart_columns = [
    'age', 'sex', 'chest_pain_type', 'resting_blood_pressure',
    'cholesterol', 'fasting_blood_sugar', 'resting_ecg', 'max_heart_rate',
    'exercise_angina', 'oldpeak', 'slope', 'major_vessels', 'thal', 'target'
]

try:
    with urlopen(heart_url, timeout=30) as response:
        heart_zip = zipfile.ZipFile(io.BytesIO(response.read()))
    heart_file = next(name for name in heart_zip.namelist() if name.endswith('processed.cleveland.data'))
    heart = pd.read_csv(heart_zip.open(heart_file), header=None, names=heart_columns, na_values='?')
except Exception as error:
    raise RuntimeError('Could not download Heart Disease UCI. Check your internet connection and rerun this cell.') from error

heart = heart.dropna().copy()
heart['target'] = (heart['target'] > 0).astype(int)
X_heart = heart.drop(columns='target')
y_heart = heart['target']
X_heart_train, X_heart_test, y_heart_train, y_heart_test = train_test_split(
    X_heart, y_heart, test_size=0.2, random_state=RANDOM_STATE, stratify=y_heart
)
print('Heart dataset rows after removing missing values:', len(heart))
print('Train rows:', len(X_heart_train), '| Test rows:', len(X_heart_test))
heart.head()

Heart dataset rows after removing missing values: 297
Train rows: 237 | Test rows: 60


,age,sex,chest_pain_type,resting_blood_pressure,cholesterol,fasting_blood_sugar,resting_ecg,max_heart_rate,exercise_angina,oldpeak,slope,major_vessels,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 2. Intel Image Classification dataset

The dataset contains six outdoor-scene categories. We read labeled images from the training directory, resize each to 24 × 24 pixels, flatten RGB values into numeric features, and make a stratified 80/20 split. The image cap is a practical speed setting, not a requirement of the dataset; raise it or set it to `None` for a larger experiment.


In [ ]:
MAX_IMAGES_PER_CLASS = 150  # Set to None to use every image in each class.
IMAGE_SIZE = (24, 24)
intel_dir = NOTEBOOK_DIR / 'intel-image-classification' / 'seg_train' / 'seg_train'
if not intel_dir.exists():
    raise FileNotFoundError(
        f'Intel images not found at {intel_dir}. Extract the Kaggle dataset so this folder contains one folder per class.'
    )

image_extensions = {'.jpg', '.jpeg', '.png'}
image_features = []
image_labels = []
image_rng = np.random.default_rng(RANDOM_STATE)
for class_folder in sorted(path for path in intel_dir.iterdir() if path.is_dir()):
    image_paths = sorted(path for path in class_folder.iterdir() if path.suffix.lower() in image_extensions)
    if MAX_IMAGES_PER_CLASS is not None and len(image_paths) > MAX_IMAGES_PER_CLASS:
        chosen_indices = image_rng.choice(len(image_paths), size=MAX_IMAGES_PER_CLASS, replace=False)
        image_paths = [image_paths[index] for index in chosen_indices]
    for image_path in image_paths:
        with Image.open(image_path) as image:
            resized = image.convert('RGB').resize(IMAGE_SIZE)
            image_features.append(np.asarray(resized, dtype=np.float32).reshape(-1) / 255.0)
            image_labels.append(class_folder.name)

X_image = np.asarray(image_features, dtype=np.float32)
y_image = np.asarray(image_labels)
if len(np.unique(y_image)) < 2:
    raise ValueError('At least two class folders with images are needed.')
X_image_train, X_image_test, y_image_train, y_image_test = train_test_split(
    X_image, y_image, test_size=0.2, random_state=RANDOM_STATE, stratify=y_image
)
print('Image count:', len(X_image), '| Classes:', sorted(np.unique(y_image)))
print('Image feature count:', X_image.shape[1])
print('Train rows:', len(X_image_train), '| Test rows:', len(X_image_test))

## 3. Decision Tree, Random Forest, and SVM comparison

We measure test accuracy and time spent fitting each model on the training data. The Random Forest uses scikit-learn defaults (`n_estimators=100`); the SVM uses an RBF kernel and standardizes its inputs. Timing depends on your computer.


In [ ]:
def compare_models(X_train, X_test, y_train, y_test):
    models = {
        'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'Random Forest (default)': RandomForestClassifier(random_state=RANDOM_STATE),
        'SVM (RBF)': make_pipeline(StandardScaler(), SVC()),
    }
    rows = []
    for name, model in models.items():
        start = time.perf_counter()
        model.fit(X_train, y_train)
        training_seconds = time.perf_counter() - start
        accuracy = accuracy_score(y_test, model.predict(X_test))
        rows.append({'Model': name, 'Test accuracy': accuracy, 'Training time (seconds)': training_seconds})
    return pd.DataFrame(rows)

heart_comparison = compare_models(X_heart_train, X_heart_test, y_heart_train, y_heart_test)
image_comparison = compare_models(X_image_train, X_image_test, y_image_train, y_image_test)

print('Heart Disease UCI')
display(heart_comparison.round(4))
print('Intel Image Classification')
display(image_comparison.round(4))
heart_comparison.to_csv(RESULTS_DIR / 'heart_model_comparison.csv', index=False)
image_comparison.to_csv(RESULTS_DIR / 'intel_model_comparison.csv', index=False)

## 4. Effect of the number of trees

The following experiment uses the requested tree counts and plots test accuracy. Increasing the tree count usually makes the result less dependent on the randomness of a small number of trees, but takes more training time.


In [ ]:
def run_tree_sweep(X_train, X_test, y_train, y_test, dataset_name):
    rows = []
    for tree_count in TREE_COUNTS:
        model = RandomForestClassifier(n_estimators=tree_count, random_state=RANDOM_STATE)
        start = time.perf_counter()
        model.fit(X_train, y_train)
        training_seconds = time.perf_counter() - start
        accuracy = accuracy_score(y_test, model.predict(X_test))
        rows.append({'n_estimators': tree_count, 'Test accuracy': accuracy, 'Training time (seconds)': training_seconds})
    results = pd.DataFrame(rows)
    results.to_csv(RESULTS_DIR / f'{dataset_name}_tree_sweep.csv', index=False)
    plt.figure(figsize=(7, 4))
    plt.plot(results['n_estimators'], results['Test accuracy'], marker='o')
    plt.xticks(TREE_COUNTS)
    plt.xlabel('Number of trees (n_estimators)')
    plt.ylabel('Test accuracy')
    plt.title(f'{dataset_name}: test accuracy vs number of trees')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'{dataset_name}_accuracy_vs_trees.png', dpi=150)
    plt.show()
    return results

heart_sweep = run_tree_sweep(X_heart_train, X_heart_test, y_heart_train, y_heart_test, 'heart')
image_sweep = run_tree_sweep(X_image_train, X_image_test, y_image_train, y_image_test, 'intel_images')
print('Heart sweep results')
display(heart_sweep.round(4))
print('Image sweep results')
display(image_sweep.round(4))

## 5. Simple stability check

We train three 100-tree forests with different random seeds. A smaller standard deviation of their test accuracies means the scores changed less across these runs. This is a small illustrative check, not a formal statistical test.


In [ ]:
def check_forest_stability(X_train, X_test, y_train, y_test, dataset_name):
    scores = []
    for seed in [0, 1, 2]:
        model = RandomForestClassifier(n_estimators=100, random_state=seed)
        model.fit(X_train, y_train)
        scores.append(accuracy_score(y_test, model.predict(X_test)))
    return {
        'Dataset': dataset_name,
        'Accuracies across seeds': scores,
        'Mean accuracy': np.mean(scores),
        'Standard deviation': np.std(scores),
    }

stability_results = pd.DataFrame([
    check_forest_stability(X_heart_train, X_heart_test, y_heart_train, y_heart_test, 'Heart Disease'),
    check_forest_stability(X_image_train, X_image_test, y_image_train, y_image_test, 'Intel Images'),
])
display(stability_results.round(4))
stability_results.to_csv(RESULTS_DIR / 'forest_stability.csv', index=False)

## 6. Results and discussion (report)

Use the comparison and sweep tables above as your results. Do not copy example numbers from elsewhere: exact scores and training times depend on the sampled images, software, and computer.

- **Number of trees:** Compare the accuracy values and plot. More trees often improve or stabilize predictions at first, then gains become small; accuracy need not rise at every step because it is measured on a finite test set. More trees require more training time.
- **Decision Tree vs Random Forest:** The forest combines many randomized trees and commonly generalizes better than one tree, which can vary greatly with its training sample. It usually takes longer to fit than one tree. Use the measured tables to state whether that happened here.
- **SVM comparison:** Report which of the three models has the highest test accuracy for each dataset, and compare the fit times. The SVM is scaled here; the tree models do not require scaling.
- **Randomness and stability:** The forest standard-deviation table compares three random seeds. Lower variation suggests more stable scores. The ensemble reduces the influence of any one tree, though it cannot guarantee higher accuracy on every dataset.

### Reproduction

1. Install `numpy`, `pandas`, `matplotlib`, `scikit-learn`, and `pillow` in your notebook environment.
2. Extract the Kaggle Intel Image Classification dataset so the path shown at the start of the image-loading cell exists. Internet access is needed to download Heart Disease UCI.
3. Open this notebook and choose **Run All**. The result tables (CSV) and accuracy plots (PNG) are saved in `random_forest_results`.
4. To share the code on GitHub, create a repository, upload this notebook and the results you wish to share, then replace this line with the repository URL: **GitHub repository: add your link here.** Do not upload the full image dataset unless its license and size permit it.
